In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import os
import pickle

c:\Users\ADMIN\anaconda3\envs\fitness-tracker\Lib\site-packages\sklearn\utils\__init__.py:15: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.26.0)
  from scipy.sparse import issparse


In [2]:
#1.READ DATA.
data = pd.read_pickle("../../data/processed/data_processed.pkl")
train_df = pd.read_pickle("../../data/processed/train.pkl")
val_df = pd.read_pickle("../../data/processed/val.pkl")
test_df = pd.read_pickle("../../data/processed/test.pkl")

In [3]:
#2.FUNCTION.

def safe_corr(a, b, eps=1e-8):
    #eps ~ 0, tránh sài số 0.
    #Tính tương quan giữa các trục acc x-y-z và gyr x-y-z
    #Nếu kết quả ra NaN thì quy về 0 vì nó không có ý nghĩa khi so với trục khác.
    if np.std(a) < eps or np.std(b) < eps:
        return 0.0 #float
    return np.corrcoef(a, b)[0, 1] # return corr(a,b)


def extract_window_features(df, window_size=10):
    #mỗi window là 10 (Mốc là 2s để đánh giá.).
    # Chỉ giữ những feature giúp phân biệt activity tốt nhất
    # -> tránh overfitting

    #Ds cột cần xử lý.
    sensor_cols = [
        'acc_x', 'acc_y', 'acc_z',
        'gyr_x', 'gyr_y', 'gyr_z'
    ]

    features_list = []

    #Duyệt theo từng người, set.
    for (user_id, set_id), group in df.groupby(['user_id', 'set']):
        n_windows = len(group) // window_size 
        #Duyệt qua từng window trong 1 set.
        for i in range(n_windows):
            start = i * window_size
            end = start + window_size
            window = group.iloc[start:end]

            #Ko đủ mẫu thì bỏ qua -> tránh lỗi tính toán.
            if len(window) < window_size:
                continue
            
            #Dict.
            window_features = {
                'user_id': user_id,
                'set': set_id,
                'label': window['label'].iloc[0],
                'weight': window['weight'].iloc[0],
                'height': window['height'].iloc[0],
                'age': window['age'].iloc[0],
                'gender': window['gender'].iloc[0]
            }

            #Duyệt qua từng cột -> trích suất đặc trưng (4 cột đại diện) -> tránh overfitting.
            #Đánh giá riêng lẻ trên x,y,z -> Chuyển động theo hướng nào.
            for col in sensor_cols:
                values = window[col].values

                window_features[f'{col}_mean'] = np.mean(values) # vị trí trung tâm. -> giúp phân biệt hướng.
                window_features[f'{col}_std'] = np.std(values) # độ biến động  -> giúp phân biệt tĩnh/ động
                window_features[f'{col}_rms'] = np.sqrt(np.mean(values ** 2)) #cường độ thật -> đo cường độ.
                window_features[f'{col}_energy'] = np.mean(values ** 2) #công suất trong 2s.

            # Gộp 3 trục -> giúp ko phụ thuộc vào hướng đặt điện thoại.
            acc_mag = np.sqrt(
                window['acc_x']**2 +
                window['acc_y']**2 +
                window['acc_z']**2
            )

            gyr_mag = np.sqrt(
                window['gyr_x']**2 +
                window['gyr_y']**2 +
                window['gyr_z']**2
            )

            # Đánh giá magnitude -> Chuyển động mạnh như nào? (Tránh TH xoay điện thoại).
            window_features['acc_mag_mean'] = np.mean(acc_mag)
            window_features['acc_mag_std'] = np.std(acc_mag)
            window_features['acc_mag_rms'] = np.sqrt(np.mean(acc_mag ** 2))

            window_features['gyr_mag_mean'] = np.mean(gyr_mag)
            window_features['gyr_mag_std'] = np.std(gyr_mag)
            window_features['gyr_mag_rms'] = np.sqrt(np.mean(gyr_mag ** 2))

            features_list.append(window_features) 
    #Những feature trên giúp dự đoán xem trong 2s chuyển động đó là gì (chạy,đi bộ,...)
    return pd.DataFrame(features_list)
#OUTCOME: Hàm safe_corr() là để tính tương quan giữa các trục x-y-z của acc và gyr(chỉ 2 đơn vị này).
#         Hàm extract_window_features, lấy 10 mẫu cho mỗi lần -> giảm tgian tính toán feature -> sài std, mean, RSM, energy:
#         Tính 2 lần: 1 lần là cho từng trục riêng lẻ, 1 lần là tổng quát cả 3 trục (ko phụ thuộc vào hướng như riêng lẻ) giúp phân biệt rõ hơn.

In [4]:
#EXTRACT FEATURE (DATA TRANSFORM).
print("\nExtracting features from windows...")
WINDOW_SIZE = 10  # 10 samples × 200ms = 2 seconds window

features_data = extract_window_features(data, window_size=WINDOW_SIZE)
features_train = extract_window_features(train_df, window_size=WINDOW_SIZE)
features_val = extract_window_features(val_df, window_size=WINDOW_SIZE)
features_test = extract_window_features(test_df, window_size=WINDOW_SIZE)


Extracting features from windows...


In [5]:
features_data

,user_id,set,label,weight,height,age,gender,acc_x_mean,acc_x_std,acc_x_rms,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,1,1,0,102,188,46,1,-0.022457,0.104458,0.106845,...,0.220659,0.372995,0.433377,0.187815,0.994623,0.233214,1.021599,1.317644,0.748152,1.515228
1,1,1,0,102,188,46,1,0.009804,0.141052,0.141393,...,0.100344,0.210554,0.233242,0.054402,1.025846,0.169824,1.039807,1.417483,0.834565,1.644919
2,1,1,0,102,188,46,1,0.037213,0.135651,0.140662,...,0.180376,0.398288,0.437228,0.191169,1.040282,0.183667,1.056371,1.644861,0.759682,1.811818
3,1,1,0,102,188,46,1,0.014949,0.074032,0.075526,...,0.284251,0.472306,0.551246,0.303872,0.990540,0.192030,1.008982,1.543851,0.663441,1.680366
4,1,1,0,102,188,46,1,0.041596,0.151239,0.156855,...,0.058833,0.335170,0.340294,0.115800,1.012775,0.166742,1.026410,1.613391,0.746819,1.777855
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13983,24,353,2,74,173,18,0,-0.192117,0.232299,0.301449,...,-0.151122,1.406393,1.414489,2.000779,1.131568,0.246099,1.158020,1.936990,0.799708,2.095582
13984,24,353,2,74,173,18,0,-0.236467,0.225630,0.326842,...,-0.115867,1.386607,1.391439,1.936104,1.094722,0.408589,1.168487,2.022268,0.726566,2.148829
13985,24,353,2,74,173,18,0,-0.209301,0.203869,0.292180,...,-0.083266,1.334841,1.337435,1.788734,1.087030,0.347608,1.141256,1.984320,0.482324,2.042098
13986,24,353,2,74,173,18,0,-0.247126,0.215143,0.327655,...,-0.072532,1.301314,1.303334,1.698680,1.098966,0.191196,1.115474,1.908579,0.753518,2.051941


In [6]:
features_train

,user_id,set,label,weight,height,age,gender,acc_x_mean,acc_x_std,acc_x_rms,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,1,1,0,102,188,46,1,-0.022457,0.104458,0.106845,...,0.220659,0.372995,0.433377,0.187815,0.994623,0.233214,1.021599,1.317644,0.748152,1.515228
1,1,1,0,102,188,46,1,0.009804,0.141052,0.141393,...,0.100344,0.210554,0.233242,0.054402,1.025846,0.169824,1.039807,1.417483,0.834565,1.644919
2,1,1,0,102,188,46,1,0.037213,0.135651,0.140662,...,0.180376,0.398288,0.437228,0.191169,1.040282,0.183667,1.056371,1.644861,0.759682,1.811818
3,1,1,0,102,188,46,1,0.014949,0.074032,0.075526,...,0.284251,0.472306,0.551246,0.303872,0.990540,0.192030,1.008982,1.543851,0.663441,1.680366
4,1,1,0,102,188,46,1,0.041596,0.151239,0.156855,...,0.058833,0.335170,0.340294,0.115800,1.012775,0.166742,1.026410,1.613391,0.746819,1.777855
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10440,18,346,2,54,164,26,0,-0.068025,0.067135,0.095575,...,0.040786,0.163686,0.168691,0.028457,1.030331,0.175195,1.045120,0.943116,0.201202,0.964339
10441,18,346,2,54,164,26,0,-0.081430,0.064823,0.104081,...,0.066729,0.149495,0.163711,0.026801,1.013765,0.206048,1.034492,1.041735,0.317350,1.089001
10442,18,346,2,54,164,26,0,-0.063070,0.067157,0.092130,...,0.079896,0.170803,0.188566,0.035557,0.986757,0.157275,0.999212,0.962485,0.266562,0.998716
10443,18,346,2,54,164,26,0,-0.067892,0.079594,0.104616,...,0.058163,0.187315,0.196137,0.038470,1.039934,0.177557,1.054983,0.951916,0.269101,0.989222


In [7]:
features_val

,user_id,set,label,weight,height,age,gender,acc_x_mean,acc_x_std,acc_x_rms,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,19,11,0,78,164,28,0,-0.207525,0.123811,0.241652,...,-0.144628,0.225833,0.268175,0.071918,1.063133,0.167564,1.076257,1.473487,0.646516,1.609083
1,19,11,0,78,164,28,0,-0.176244,0.061131,0.186545,...,-0.027269,0.233665,0.235251,0.055343,1.005034,0.259805,1.038072,0.470371,0.176219,0.502297
2,19,11,0,78,164,28,0,-0.207504,0.053093,0.214189,...,-0.024767,0.212132,0.213573,0.045613,0.991920,0.302735,1.037090,0.415511,0.177805,0.451956
3,19,11,0,78,164,28,0,-0.248827,0.091573,0.265143,...,-0.063111,0.249561,0.257417,0.066264,1.045551,0.199735,1.064458,1.207877,0.490399,1.303633
4,19,11,0,78,164,28,0,-0.216663,0.060763,0.225023,...,-0.089655,0.251136,0.266660,0.071107,1.033524,0.256661,1.064917,0.487756,0.366059,0.609841
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1940,21,350,2,52,165,24,1,-0.278689,0.080730,0.290146,...,0.123326,0.296616,0.321232,0.103190,1.047386,0.158408,1.059297,1.581893,0.754236,1.752500
1941,21,350,2,52,165,24,1,-0.257495,0.090124,0.272811,...,0.185220,0.270118,0.327521,0.107270,0.974041,0.215425,0.997579,1.476576,0.839732,1.698654
1942,21,350,2,52,165,24,1,-0.313846,0.098671,0.328992,...,0.137498,0.376403,0.400730,0.160585,1.054926,0.248217,1.083735,1.732480,0.661034,1.854306
1943,21,350,2,52,165,24,1,-0.248173,0.070854,0.258089,...,0.159190,0.283895,0.325481,0.105938,1.039778,0.257272,1.071133,1.605642,0.964587,1.873103


In [8]:
features_test

,user_id,set,label,weight,height,age,gender,acc_x_mean,acc_x_std,acc_x_rms,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,22,15,0,100,186,31,1,0.167575,0.203078,0.263291,...,0.300513,1.047388,1.089647,1.187330,1.091826,0.277969,1.126655,1.833520,1.121698,2.149419
1,22,15,0,100,186,31,1,0.163702,0.174087,0.238966,...,0.146734,0.854720,0.867224,0.752077,1.085309,0.239589,1.111440,1.671124,0.526611,1.752135
2,22,15,0,100,186,31,1,0.271278,0.116935,0.295407,...,-0.141600,0.559155,0.576806,0.332705,1.073309,0.327395,1.122131,1.580448,0.721405,1.737308
3,22,15,0,100,186,31,1,0.161363,0.180530,0.242135,...,0.543324,1.175191,1.294711,1.676276,1.077606,0.250286,1.106290,1.990974,1.239026,2.345029
4,22,15,0,100,186,31,1,0.249936,0.087628,0.264852,...,-0.022119,0.591241,0.591654,0.350055,1.041494,0.297763,1.083224,1.671798,0.762934,1.837655
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1593,24,353,2,74,173,18,0,-0.192117,0.232299,0.301449,...,-0.151122,1.406393,1.414489,2.000779,1.131568,0.246099,1.158020,1.936990,0.799708,2.095582
1594,24,353,2,74,173,18,0,-0.236467,0.225630,0.326842,...,-0.115867,1.386607,1.391439,1.936104,1.094722,0.408589,1.168487,2.022268,0.726566,2.148829
1595,24,353,2,74,173,18,0,-0.209301,0.203869,0.292180,...,-0.083266,1.334841,1.337435,1.788734,1.087030,0.347608,1.141256,1.984320,0.482324,2.042098
1596,24,353,2,74,173,18,0,-0.247126,0.215143,0.327655,...,-0.072532,1.301314,1.303334,1.698680,1.098966,0.191196,1.115474,1.908579,0.753518,2.051941


In [9]:
print(f"NaN values in train: {features_train.isna().sum().sum()}")
print(f"Inf values in train: {np.isinf(features_train.select_dtypes(include=[np.number])).sum().sum()}")

NaN values in train: 0
Inf values in train: 0


In [10]:
features_data = features_data.replace([np.inf, -np.inf], np.nan)
features_train = features_train.replace([np.inf, -np.inf], np.nan)
features_val = features_val.replace([np.inf, -np.inf], np.nan)
features_test = features_test.replace([np.inf, -np.inf], np.nan)

In [11]:
features_data

,user_id,set,label,weight,height,age,gender,acc_x_mean,acc_x_std,acc_x_rms,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,1,1,0,102,188,46,1,-0.022457,0.104458,0.106845,...,0.220659,0.372995,0.433377,0.187815,0.994623,0.233214,1.021599,1.317644,0.748152,1.515228
1,1,1,0,102,188,46,1,0.009804,0.141052,0.141393,...,0.100344,0.210554,0.233242,0.054402,1.025846,0.169824,1.039807,1.417483,0.834565,1.644919
2,1,1,0,102,188,46,1,0.037213,0.135651,0.140662,...,0.180376,0.398288,0.437228,0.191169,1.040282,0.183667,1.056371,1.644861,0.759682,1.811818
3,1,1,0,102,188,46,1,0.014949,0.074032,0.075526,...,0.284251,0.472306,0.551246,0.303872,0.990540,0.192030,1.008982,1.543851,0.663441,1.680366
4,1,1,0,102,188,46,1,0.041596,0.151239,0.156855,...,0.058833,0.335170,0.340294,0.115800,1.012775,0.166742,1.026410,1.613391,0.746819,1.777855
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13983,24,353,2,74,173,18,0,-0.192117,0.232299,0.301449,...,-0.151122,1.406393,1.414489,2.000779,1.131568,0.246099,1.158020,1.936990,0.799708,2.095582
13984,24,353,2,74,173,18,0,-0.236467,0.225630,0.326842,...,-0.115867,1.386607,1.391439,1.936104,1.094722,0.408589,1.168487,2.022268,0.726566,2.148829
13985,24,353,2,74,173,18,0,-0.209301,0.203869,0.292180,...,-0.083266,1.334841,1.337435,1.788734,1.087030,0.347608,1.141256,1.984320,0.482324,2.042098
13986,24,353,2,74,173,18,0,-0.247126,0.215143,0.327655,...,-0.072532,1.301314,1.303334,1.698680,1.098966,0.191196,1.115474,1.908579,0.753518,2.051941


In [12]:
metadata_cols = ['user_id', 'set', 'label', 'weight', 'height', 'age', 'gender']

In [13]:
feature_cols = [col for col in features_train.columns if col not in metadata_cols]

In [14]:
feature_cols

['acc_x_mean',
 'acc_x_std',
 'acc_x_rms',
 'acc_x_energy',
 'acc_y_mean',
 'acc_y_std',
 'acc_y_rms',
 'acc_y_energy',
 'acc_z_mean',
 'acc_z_std',
 'acc_z_rms',
 'acc_z_energy',
 'gyr_x_mean',
 'gyr_x_std',
 'gyr_x_rms',
 'gyr_x_energy',
 'gyr_y_mean',
 'gyr_y_std',
 'gyr_y_rms',
 'gyr_y_energy',
 'gyr_z_mean',
 'gyr_z_std',
 'gyr_z_rms',
 'gyr_z_energy',
 'acc_mag_mean',
 'acc_mag_std',
 'acc_mag_rms',
 'gyr_mag_mean',
 'gyr_mag_std',
 'gyr_mag_rms']

In [15]:
imputer = SimpleImputer(strategy='median') #Median ít ảnh hưởng bởi outlier.

In [16]:
imputer.fit(features_train[feature_cols])

SimpleImputer(strategy='median')

In [17]:
features_data[feature_cols] = imputer.transform(features_data[feature_cols])
features_train[feature_cols] = imputer.transform(features_train[feature_cols])
features_val[feature_cols] = imputer.transform(features_val[feature_cols])
features_test[feature_cols] = imputer.transform(features_test[feature_cols])

In [18]:
features_data[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,-0.022457,0.104458,0.106845,0.011416,0.974910,0.231350,1.001984,1.003972,-0.067165,0.154159,...,0.220659,0.372995,0.433377,0.187815,0.994623,0.233214,1.021599,1.317644,0.748152,1.515228
1,0.009804,0.141052,0.141393,0.019992,0.982654,0.163062,0.996091,0.992198,-0.221017,0.141991,...,0.100344,0.210554,0.233242,0.054402,1.025846,0.169824,1.039807,1.417483,0.834565,1.644919
2,0.037213,0.135651,0.140662,0.019786,1.016568,0.186068,1.033456,1.068031,-0.119377,0.117692,...,0.180376,0.398288,0.437228,0.191169,1.040282,0.183667,1.056371,1.644861,0.759682,1.811818
3,0.014949,0.074032,0.075526,0.005704,0.977858,0.195575,0.997224,0.994457,-0.117630,0.063621,...,0.284251,0.472306,0.551246,0.303872,0.990540,0.192030,1.008982,1.543851,0.663441,1.680366
4,0.041596,0.151239,0.156855,0.024603,0.964831,0.161974,0.978333,0.957135,-0.231735,0.134452,...,0.058833,0.335170,0.340294,0.115800,1.012775,0.166742,1.026410,1.613391,0.746819,1.777855
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13983,-0.192117,0.232299,0.301449,0.090872,1.065065,0.251376,1.094328,1.197554,-0.040070,0.225783,...,-0.151122,1.406393,1.414489,2.000779,1.131568,0.246099,1.158020,1.936990,0.799708,2.095582
13984,-0.236467,0.225630,0.326842,0.106826,1.038689,0.384251,1.107485,1.226524,-0.022727,0.177472,...,-0.115867,1.386607,1.391439,1.936104,1.094722,0.408589,1.168487,2.022268,0.726566,2.148829
13985,-0.209301,0.203869,0.292180,0.085369,1.038013,0.323419,1.087230,1.182070,-0.047367,0.181057,...,-0.083266,1.334841,1.337435,1.788734,1.087030,0.347608,1.141256,1.984320,0.482324,2.042098
13986,-0.247126,0.215143,0.327655,0.107358,1.015640,0.168569,1.029534,1.059940,-0.005894,0.277400,...,-0.072532,1.301314,1.303334,1.698680,1.098966,0.191196,1.115474,1.908579,0.753518,2.051941


In [19]:
features_train[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,-0.022457,0.104458,0.106845,0.011416,0.974910,0.231350,1.001984,1.003972,-0.067165,0.154159,...,0.220659,0.372995,0.433377,0.187815,0.994623,0.233214,1.021599,1.317644,0.748152,1.515228
1,0.009804,0.141052,0.141393,0.019992,0.982654,0.163062,0.996091,0.992198,-0.221017,0.141991,...,0.100344,0.210554,0.233242,0.054402,1.025846,0.169824,1.039807,1.417483,0.834565,1.644919
2,0.037213,0.135651,0.140662,0.019786,1.016568,0.186068,1.033456,1.068031,-0.119377,0.117692,...,0.180376,0.398288,0.437228,0.191169,1.040282,0.183667,1.056371,1.644861,0.759682,1.811818
3,0.014949,0.074032,0.075526,0.005704,0.977858,0.195575,0.997224,0.994457,-0.117630,0.063621,...,0.284251,0.472306,0.551246,0.303872,0.990540,0.192030,1.008982,1.543851,0.663441,1.680366
4,0.041596,0.151239,0.156855,0.024603,0.964831,0.161974,0.978333,0.957135,-0.231735,0.134452,...,0.058833,0.335170,0.340294,0.115800,1.012775,0.166742,1.026410,1.613391,0.746819,1.777855
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10440,-0.068025,0.067135,0.095575,0.009135,1.010759,0.169300,1.024839,1.050295,-0.053481,0.173162,...,0.040786,0.163686,0.168691,0.028457,1.030331,0.175195,1.045120,0.943116,0.201202,0.964339
10441,-0.081430,0.064823,0.104081,0.010833,0.998630,0.206582,1.019773,1.039937,-0.038340,0.133919,...,0.066729,0.149495,0.163711,0.026801,1.013765,0.206048,1.034492,1.041735,0.317350,1.089001
10442,-0.063070,0.067157,0.092130,0.008488,0.970171,0.158145,0.982976,0.966242,-0.012686,0.153408,...,0.079896,0.170803,0.188566,0.035557,0.986757,0.157275,0.999212,0.962485,0.266562,0.998716
10443,-0.067892,0.079594,0.104616,0.010945,1.024836,0.174411,1.039571,1.080708,-0.033746,0.142117,...,0.058163,0.187315,0.196137,0.038470,1.039934,0.177557,1.054983,0.951916,0.269101,0.989222


In [20]:
features_val[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,-0.207525,0.123811,0.241652,0.058396,1.024647,0.165144,1.037870,1.077174,-0.034049,0.146972,...,-0.144628,0.225833,0.268175,0.071918,1.063133,0.167564,1.076257,1.473487,0.646516,1.609083
1,-0.176244,0.061131,0.186545,0.034799,0.972936,0.271129,1.010008,1.020116,0.114290,0.098061,...,-0.027269,0.233665,0.235251,0.055343,1.005034,0.259805,1.038072,0.470371,0.176219,0.502297
2,-0.207504,0.053093,0.214189,0.045877,0.957750,0.318774,1.009407,1.018902,0.084307,0.060568,...,-0.024767,0.212132,0.213573,0.045613,0.991920,0.302735,1.037090,0.415511,0.177805,0.451956
3,-0.248827,0.091573,0.265143,0.070301,1.001173,0.195785,1.020137,1.040680,0.031257,0.145302,...,-0.063111,0.249561,0.257417,0.066264,1.045551,0.199735,1.064458,1.207877,0.490399,1.303633
4,-0.216663,0.060763,0.225023,0.050635,0.999066,0.263748,1.033294,1.067696,0.076051,0.099662,...,-0.089655,0.251136,0.266660,0.071107,1.033524,0.256661,1.064917,0.487756,0.366059,0.609841
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1940,-0.278689,0.080730,0.290146,0.084185,0.978435,0.175011,0.993964,0.987964,-0.093575,0.202992,...,0.123326,0.296616,0.321232,0.103190,1.047386,0.158408,1.059297,1.581893,0.754236,1.752500
1941,-0.257495,0.090124,0.272811,0.074426,0.892482,0.238679,0.923846,0.853491,-0.146591,0.213911,...,0.185220,0.270118,0.327521,0.107270,0.974041,0.215425,0.997579,1.476576,0.839732,1.698654
1942,-0.313846,0.098671,0.328992,0.108236,0.958839,0.239631,0.988330,0.976796,-0.134990,0.266885,...,0.137498,0.376403,0.400730,0.160585,1.054926,0.248217,1.083735,1.732480,0.661034,1.854306
1943,-0.248173,0.070854,0.258089,0.066610,0.976851,0.257903,1.010323,1.020753,-0.084552,0.229814,...,0.159190,0.283895,0.325481,0.105938,1.039778,0.257272,1.071133,1.605642,0.964587,1.873103


In [21]:
features_test[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,0.167575,0.203078,0.263291,0.069322,1.035393,0.261245,1.067842,1.140287,-0.129797,0.207110,...,0.300513,1.047388,1.089647,1.187330,1.091826,0.277969,1.126655,1.833520,1.121698,2.149419
1,0.163702,0.174087,0.238966,0.057105,1.034604,0.233704,1.060671,1.125024,-0.131826,0.189187,...,0.146734,0.854720,0.867224,0.752077,1.085309,0.239589,1.111440,1.671124,0.526611,1.752135
2,0.271278,0.116935,0.295407,0.087265,0.997363,0.323347,1.048468,1.099285,-0.238707,0.125088,...,-0.141600,0.559155,0.576806,0.332705,1.073309,0.327395,1.122131,1.580448,0.721405,1.737308
3,0.161363,0.180530,0.242135,0.058629,1.037655,0.242559,1.065628,1.135563,-0.061993,0.160752,...,0.543324,1.175191,1.294711,1.676276,1.077606,0.250286,1.106290,1.990974,1.239026,2.345029
4,0.249936,0.087628,0.264852,0.070147,0.961828,0.330523,1.017035,1.034359,-0.237928,0.110713,...,-0.022119,0.591241,0.591654,0.350055,1.041494,0.297763,1.083224,1.671798,0.762934,1.837655
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1593,-0.192117,0.232299,0.301449,0.090872,1.065065,0.251376,1.094328,1.197554,-0.040070,0.225783,...,-0.151122,1.406393,1.414489,2.000779,1.131568,0.246099,1.158020,1.936990,0.799708,2.095582
1594,-0.236467,0.225630,0.326842,0.106826,1.038689,0.384251,1.107485,1.226524,-0.022727,0.177472,...,-0.115867,1.386607,1.391439,1.936104,1.094722,0.408589,1.168487,2.022268,0.726566,2.148829
1595,-0.209301,0.203869,0.292180,0.085369,1.038013,0.323419,1.087230,1.182070,-0.047367,0.181057,...,-0.083266,1.334841,1.337435,1.788734,1.087030,0.347608,1.141256,1.984320,0.482324,2.042098
1596,-0.247126,0.215143,0.327655,0.107358,1.015640,0.168569,1.029534,1.059940,-0.005894,0.277400,...,-0.072532,1.301314,1.303334,1.698680,1.098966,0.191196,1.115474,1.908579,0.753518,2.051941


In [22]:
print(f"Number of features to scale: {len(feature_cols)}")

Number of features to scale: 30


In [23]:
scaler = StandardScaler()

In [24]:
scaler.fit(features_train[feature_cols])

StandardScaler()

In [25]:
features_data[feature_cols] = scaler.transform(features_data[feature_cols])
features_train[feature_cols] = scaler.transform(features_train[feature_cols])
features_val[feature_cols] = scaler.transform(features_val[feature_cols])
features_test[feature_cols] = scaler.transform(features_test[feature_cols])

In [26]:
features_data[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,-0.281720,0.222229,-0.921847,-0.637937,0.512744,0.356033,0.489036,0.423749,0.071810,0.373354,...,2.200803,0.106204,0.237899,-0.161665,-0.681967,0.389626,-0.438542,0.504030,0.902951,0.576816
1,-0.183346,0.627491,-0.751731,-0.590894,0.534999,-0.016819,0.470816,0.397116,-0.316518,0.265261,...,0.934940,-0.313607,-0.273281,-0.427191,-0.165604,0.037807,-0.239664,0.621227,1.127670,0.716182
2,-0.099768,0.567668,-0.755328,-0.592024,0.632464,0.108796,0.586340,0.568644,-0.059976,0.049393,...,1.776978,0.171572,0.247738,-0.154991,0.073141,0.114635,-0.058758,0.888137,0.932937,0.895533
3,-0.167655,-0.114725,-1.076063,-0.669268,0.521217,0.160704,0.474320,0.402226,-0.055565,-0.430971,...,2.869879,0.362864,0.538958,0.069316,-0.749494,0.161052,-0.576340,0.769566,0.682660,0.754274
4,-0.086402,0.740297,-0.675596,-0.565598,0.483779,-0.022756,0.415911,0.317808,-0.343569,0.198283,...,0.498194,0.008450,0.000150,-0.304993,-0.381765,0.020701,-0.385997,0.851196,0.899485,0.859036
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13983,-0.799066,1.637989,0.036400,-0.202088,0.771842,0.465373,0.774543,0.861611,0.140198,1.009659,...,-1.710789,2.776914,2.743838,3.446580,1.582843,0.461136,1.051458,1.231056,1.037025,1.200466
13984,-0.934303,1.564136,0.161434,-0.114574,0.696039,1.190860,0.815221,0.927137,0.183973,0.580469,...,-1.339859,2.725779,2.684966,3.317860,0.973490,1.362970,1.165782,1.331160,0.846817,1.257686
13985,-0.851464,1.323146,-0.009241,-0.232271,0.694095,0.858719,0.752597,0.826586,0.121779,0.612318,...,-0.996864,2.591996,2.547030,3.024558,0.846265,1.024519,0.868358,1.286615,0.211664,1.142992
13986,-0.966806,1.448000,0.165440,-0.111654,0.629798,0.013251,0.574213,0.550342,0.226459,1.468215,...,-0.883929,2.505350,2.459929,2.845329,1.043679,0.156422,0.586774,1.197705,0.916906,1.153570


In [27]:
features_train[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,-0.281720,0.222229,-0.921847,-0.637937,0.512744,0.356033,0.489036,0.423749,0.071810,0.373354,...,2.200803,0.106204,0.237899,-0.161665,-0.681967,0.389626,-0.438542,0.504030,0.902951,0.576816
1,-0.183346,0.627491,-0.751731,-0.590894,0.534999,-0.016819,0.470816,0.397116,-0.316518,0.265261,...,0.934940,-0.313607,-0.273281,-0.427191,-0.165604,0.037807,-0.239664,0.621227,1.127670,0.716182
2,-0.099768,0.567668,-0.755328,-0.592024,0.632464,0.108796,0.586340,0.568644,-0.059976,0.049393,...,1.776978,0.171572,0.247738,-0.154991,0.073141,0.114635,-0.058758,0.888137,0.932937,0.895533
3,-0.167655,-0.114725,-1.076063,-0.669268,0.521217,0.160704,0.474320,0.402226,-0.055565,-0.430971,...,2.869879,0.362864,0.538958,0.069316,-0.749494,0.161052,-0.576340,0.769566,0.682660,0.754274
4,-0.086402,0.740297,-0.675596,-0.565598,0.483779,-0.022756,0.415911,0.317808,-0.343569,0.198283,...,0.498194,0.008450,0.000150,-0.304993,-0.381765,0.020701,-0.385997,0.851196,0.899485,0.859036
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10440,-0.420670,-0.191103,-0.977343,-0.650451,0.615769,0.017241,0.559698,0.528527,0.106347,0.542178,...,0.308323,-0.434731,-0.438155,-0.478828,-0.091425,0.067614,-0.181645,0.064387,-0.519395,-0.015170
10441,-0.461546,-0.216705,-0.935456,-0.641135,0.580912,0.220799,0.544036,0.505099,0.144565,0.193548,...,0.581271,-0.471408,-0.450874,-0.482123,-0.365401,0.238852,-0.297716,0.180151,-0.217351,0.118791
10442,-0.405559,-0.190855,-0.994306,-0.653998,0.499126,-0.043665,0.430268,0.338408,0.209315,0.366685,...,0.719811,-0.416339,-0.387392,-0.464697,-0.812057,-0.031839,-0.683048,0.087123,-0.349426,0.021771
10443,-0.420265,-0.053128,-0.932821,-0.640522,0.656227,0.045146,0.605246,0.597318,0.156160,0.266375,...,0.491143,-0.373666,-0.368054,-0.458900,0.067388,0.080723,-0.073919,0.074717,-0.342825,0.011569


In [28]:
features_val[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,-0.846048,0.436548,-0.258048,-0.380233,0.655683,-0.005451,0.599986,0.589323,0.155395,0.309506,...,-1.642463,-0.274119,-0.184055,-0.392330,0.451059,0.025262,0.158440,0.686968,0.638647,0.677673
1,-0.750663,-0.257591,-0.529399,-0.509671,0.507072,0.573219,0.513843,0.460264,0.529805,-0.125013,...,-0.407698,-0.253878,-0.268150,-0.425318,-0.509783,0.537207,-0.258623,-0.490550,-0.584363,-0.511682
2,-0.845986,-0.346605,-0.393276,-0.448903,0.463427,0.833361,0.511984,0.457518,0.454128,-0.458091,...,-0.381382,-0.309528,-0.323519,-0.444682,-0.726666,0.775474,-0.269350,-0.554948,-0.580240,-0.565779
3,-0.971992,0.079534,-0.142377,-0.314929,0.588222,0.161849,0.545161,0.506778,0.320228,0.294672,...,-0.784805,-0.212797,-0.211533,-0.403583,0.160280,0.203812,0.029566,0.375179,0.232662,0.349435
4,-0.873914,-0.261669,-0.339931,-0.422803,0.582167,0.532921,0.585838,0.567886,0.433290,-0.110787,...,-1.064077,-0.208726,-0.187926,-0.393943,-0.038612,0.519758,0.034580,-0.470142,-0.090683,-0.396115
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1940,-1.063049,-0.040545,-0.019259,-0.238769,0.522875,0.048427,0.464239,0.387541,0.005149,0.807187,...,1.176742,-0.091189,-0.048537,-0.330090,0.190638,-0.025552,-0.026793,0.814221,0.918773,0.831789
1941,-0.998421,0.063490,-0.104618,-0.292301,0.275853,0.396050,0.247450,0.083377,-0.128664,0.904186,...,1.827942,-0.159671,-0.032476,-0.321971,-1.022357,0.290894,-0.700887,0.690594,1.141106,0.773926
1942,-1.170256,0.158140,0.172021,-0.106840,0.466558,0.401247,0.446820,0.362279,-0.099382,1.374798,...,1.325849,0.115012,0.154514,-0.215861,0.315334,0.472896,0.240112,0.990989,0.676402,0.941191
1943,-0.969996,-0.149918,-0.177110,-0.335174,0.518323,0.501009,0.514818,0.461704,0.027923,1.045468,...,1.554078,-0.124065,-0.037687,-0.324622,0.064804,0.523147,0.102476,0.842100,1.465793,0.961390


In [29]:
features_test[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,0.297749,1.314393,-0.151492,-0.320295,0.686565,0.519256,0.692654,0.732078,-0.086276,0.843772,...,3.040975,1.849103,1.914132,1.827616,0.925592,0.638020,0.708888,1.109597,1.874362,1.258320
1,0.285937,0.993333,-0.271273,-0.387314,0.684300,0.368884,0.670484,0.697555,-0.091396,0.684540,...,1.423025,1.351173,1.346024,0.961355,0.817808,0.425008,0.542708,0.918967,0.326832,0.831397
2,0.613969,0.360403,0.006648,-0.221870,0.577271,0.858327,0.632754,0.639337,-0.361168,0.115094,...,-1.610610,0.587317,0.604244,0.126701,0.619349,0.912335,0.659483,0.812525,0.833395,0.815464
3,0.278806,1.064685,-0.255669,-0.378951,0.693068,0.417233,0.685809,0.721394,0.084863,0.431932,...,5.595643,2.179398,2.437903,2.800738,0.690412,0.484375,0.486459,1.294425,2.179475,1.468523
4,0.548892,0.035845,-0.143807,-0.315773,0.475149,0.897510,0.535568,0.492482,-0.359202,-0.012610,...,-0.353519,0.670239,0.642170,0.161232,0.093196,0.747879,0.234529,0.919757,0.941394,0.923298
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1593,-0.799066,1.637989,0.036400,-0.202088,0.771842,0.465373,0.774543,0.861611,0.140198,1.009659,...,-1.710789,2.776914,2.743838,3.446580,1.582843,0.461136,1.051458,1.231056,1.037025,1.200466
1594,-0.934303,1.564136,0.161434,-0.114574,0.696039,1.190860,0.815221,0.927137,0.183973,0.580469,...,-1.339859,2.725779,2.684966,3.317860,0.973490,1.362970,1.165782,1.331160,0.846817,1.257686
1595,-0.851464,1.323146,-0.009241,-0.232271,0.694095,0.858719,0.752597,0.826586,0.121779,0.612318,...,-0.996864,2.591996,2.547030,3.024558,0.846265,1.024519,0.868358,1.286615,0.211664,1.142992
1596,-0.966806,1.448000,0.165440,-0.111654,0.629798,0.013251,0.574213,0.550342,0.226459,1.468215,...,-0.883929,2.505350,2.459929,2.845329,1.043679,0.156422,0.586774,1.197705,0.916906,1.153570


In [30]:
assert features_train.isna().sum().sum() == 0, "Training data contains NaN!"
assert features_val.isna().sum().sum() == 0, "Validation data contains NaN!"
assert features_test.isna().sum().sum() == 0, "Test data contains NaN!"

assert np.isinf(features_train.select_dtypes(include=[np.number])).sum().sum() == 0, "Training data contains infinity!"
assert np.isinf(features_val.select_dtypes(include=[np.number])).sum().sum() == 0, "Validation data contains infinity!"
assert np.isinf(features_test.select_dtypes(include=[np.number])).sum().sum() == 0, "Test data contains infinity!"

In [31]:
print("\nLabel distribution:")
print(f"Train: {features_train['label'].value_counts().sort_index()}")
print(f"Val: {features_val['label'].value_counts().sort_index()}")
print(f"Test: {features_test['label'].value_counts().sort_index()}")


Label distribution:
Train: label
0     966
1    1135
2    2549
3     976
4    2609
5    2210
Name: count, dtype: int64
Val: label
0    184
1    229
2    467
3    189
4    413
5    463
Name: count, dtype: int64
Test: label
0    139
1    184
2    398
3    156
4    349
5    372
Name: count, dtype: int64


In [32]:
#Summary
print(f"Total features created: {len(feature_cols)}")
print(f"Window size: {WINDOW_SIZE} samples (2 seconds)")
print(f"\nDataset sizes:")
print(f"  Train: {features_train.shape[0]} samples")
print(f"  Val:   {features_val.shape[0]} samples")
print(f"  Test:  {features_test.shape[0]} samples")
print(f"  Total: {features_data.shape[0]} samples")

Total features created: 30
Window size: 10 samples (2 seconds)

Dataset sizes:
  Train: 10445 samples
  Val:   1945 samples
  Test:  1598 samples
  Total: 13988 samples
